# Kafka — produire, consommer, traiter en continu

**Formation Big Data — ANSD / Data Innovation Lab**

Jusqu'ici les données étaient là, dans un fichier. Elles vont maintenant
**arriver** : nous simulons la remontée d'actes d'état civil depuis les centres,
au fil de l'eau.

Au programme : créer un sujet, y produire des messages, les consommer, observer
les décalages, puis brancher Spark sur le flux pour alimenter MongoDB.

Le code est fourni. Gardez l'interface **Kafka UI** ouverte sur
<http://localhost:8085> : vous y verrez tout ce qui se passe.

## 1. Se connecter au courtier

In [ ]:
import json
import os
import random
import time
from datetime import datetime, timedelta

from confluent_kafka import Consumer, Producer
from confluent_kafka.admin import AdminClient, NewTopic

COURTIER = os.environ.get("KAFKA_BOOTSTRAP", "kafka:9092")
SUJET = "actes-etat-civil"

admin = AdminClient({"bootstrap.servers": COURTIER})
metadonnees = admin.list_topics(timeout=10)

print("Courtier joint :", COURTIER)
print("Sujets existants :", [s for s in metadonnees.topics if not s.startswith("__")])

## 2. Créer un sujet

Un sujet est un journal nommé. Le nombre de **partitions** se décide ici : c'est
lui qui plafonnera le parallélisme des consommateurs, et il est délicat à
modifier ensuite.

In [ ]:
nouveau = NewTopic(SUJET, num_partitions=3, replication_factor=1)

if SUJET not in metadonnees.topics:
    for nom, futur in admin.create_topics([nouveau]).items():
        futur.result()          # lève une exception en cas d'échec
        print(f"Sujet « {nom} » créé.")
else:
    print(f"Sujet « {SUJET} » déjà présent.")

description = admin.list_topics(timeout=10).topics[SUJET]
print(f"Partitions : {len(description.partitions)}")

## 3. Produire des messages

Nous simulons des déclarations arrivant des centres d'état civil.

Deux points méritent attention dans le code ci-dessous. La **clé** du message
détermine la partition : tous les messages d'un même centre iront dans la même
partition, ce qui garantit leur ordre. Et `flush()` attend que tout soit
réellement parti — sans lui, le programme peut se terminer avant l'envoi.

In [ ]:
REGIONS = ["Dakar", "Thiès", "Diourbel", "Saint-Louis", "Ziguinchor",
           "Kaolack", "Tambacounda", "Matam"]
TYPES = ["naissance", "mariage", "deces"]
NOMS = ["Diop", "Ndiaye", "Fall", "Sarr", "Ba", "Sow", "Diallo", "Gueye"]
PRENOMS = ["Mamadou", "Aminata", "Ousmane", "Fatou", "Ibrahima", "Awa"]

alea = random.Random(2026)


def evenement(numero):
    """Fabrique un acte, tel qu'il serait transmis par un centre."""
    region = alea.choice(REGIONS)
    return {
        "numero_acte": f"ACT-{numero:07d}",
        "type_acte": alea.choices(TYPES, weights=[0.72, 0.12, 0.16])[0],
        "centre": f"CEC-{region[:2].upper()}-{alea.randrange(1, 20):02d}",
        "region": region,
        "nom": alea.choice(NOMS),
        "prenom": alea.choice(PRENOMS),
        "sexe": alea.choice(["M", "F"]),
        "horodatage": datetime.now().isoformat(timespec="seconds"),
    }


def accuse(erreur, message):
    """Appelé pour chaque message, une fois son sort connu."""
    if erreur is not None:
        print("Échec d'envoi :", erreur)


producteur = Producer({"bootstrap.servers": COURTIER})

for i in range(1, 501):
    acte = evenement(i)
    producteur.produce(
        SUJET,
        key=acte["centre"],                        # détermine la partition
        value=json.dumps(acte).encode("utf-8"),
        callback=accuse,
    )
    producteur.poll(0)                             # traite les accusés en attente

producteur.flush()                                 # attend l'envoi effectif
print("500 actes envoyés.")

👉 **Ouvrez maintenant Kafka UI** (<http://localhost:8085>), onglet *Topics*,
sujet `actes-etat-civil`. Vous voyez le nombre de messages, leur répartition
entre les trois partitions, et vous pouvez les lire un à un.

## 4. Consommer

Un consommateur appartient à un **groupe**. Le groupe mémorise où il en est de
sa lecture : c'est le **décalage**.

`auto.offset.reset` indique où commencer quand le groupe n'a pas d'historique —
`earliest` pour tout relire depuis le début, `latest` pour ne prendre que la
suite.

In [ ]:
consommateur = Consumer({
    "bootstrap.servers": COURTIER,
    "group.id": "demonstration",
    "auto.offset.reset": "earliest",
})
consommateur.subscribe([SUJET])

recus, par_partition = 0, {}
depart = time.time()

while recus < 20 and time.time() - depart < 30:
    message = consommateur.poll(timeout=1.0)
    if message is None:
        continue
    if message.error():
        print("Erreur :", message.error())
        continue
    acte = json.loads(message.value())
    par_partition[message.partition()] = par_partition.get(message.partition(), 0) + 1
    if recus < 3:
        print(f"partition {message.partition()} · décalage {message.offset()} "
              f"· clé {message.key().decode()} → {acte['numero_acte']} "
              f"({acte['type_acte']})")
    recus += 1

print(f"\n{recus} messages lus, répartition par partition : {par_partition}")
consommateur.close()

> ⚠️ La boucle est **bornée** par un nombre de messages et un délai. Sans ces
> deux garde-fous, elle attendrait indéfiniment de nouveaux messages — et la
> cellule ne rendrait jamais la main.

### Le décalage appartient au groupe

Relire avec le **même** groupe reprend là où l'on s'était arrêté. Avec un
**autre** groupe, on relit tout depuis le début. C'est ce qui permet à plusieurs
services de consommer indépendamment le même flux.

In [ ]:
def compter(groupe, limite=1000, delai=10):
    """Compte les messages lus par un groupe donné, en temps borné."""
    consommateur = Consumer({
        "bootstrap.servers": COURTIER,
        "group.id": groupe,
        "auto.offset.reset": "earliest",
    })
    consommateur.subscribe([SUJET])
    total, depart = 0, time.time()
    while total < limite and time.time() - depart < delai:
        message = consommateur.poll(timeout=1.0)
        if message is not None and not message.error():
            total += 1
    consommateur.close()
    return total


print("Groupe « demonstration », deuxième lecture :", compter("demonstration"))
print("Groupe « tableau-de-bord », première lecture :", compter("tableau-de-bord"))

Le premier groupe avait déjà lu vingt messages : il ne reprend que la suite.
Le second découvre le sujet : il lit tout.

C'est le principe qui permet d'alimenter un tableau de bord et un entrepôt de
données à partir du même flux, sans que l'un gêne l'autre.

## 5. Traiter le flux avec Spark

Spark lit un sujet Kafka comme il lirait un fichier. Deux modes existent, et le
choix entre les deux est une vraie décision d'architecture.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType, StructField, StructType

spark = (
    SparkSession.builder
    .appName("kafka")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### Mode lot : lire ce qui est disponible, puis s'arrêter

C'est souvent le mode le plus adapté à un institut de statistique : traiter
toutes les heures ce qui est arrivé suffit, et c'est bien plus simple à
exploiter qu'un traitement permanent.

In [ ]:
brut = (spark.read.format("kafka")
        .option("kafka.bootstrap.servers", COURTIER)
        .option("subscribe", SUJET)
        .option("startingOffsets", "earliest")
        .load())

print(f"{brut.count()} messages lus")
brut.select("key", "topic", "partition", "offset", "timestamp").show(5)

Kafka ne connaît que des octets : la charge utile arrive dans une colonne
`value` de type binaire. C'est à nous de la décoder et d'en déclarer la
structure.

In [ ]:
schema = StructType([
    StructField("numero_acte", StringType()),
    StructField("type_acte", StringType()),
    StructField("centre", StringType()),
    StructField("region", StringType()),
    StructField("nom", StringType()),
    StructField("prenom", StringType()),
    StructField("sexe", StringType()),
    StructField("horodatage", StringType()),
])

actes = (brut
         .select(F.from_json(F.col("value").cast("string"), schema).alias("acte"),
                 F.col("partition"), F.col("offset"))
         .select("acte.*", "partition", "offset"))

actes.show(5, truncate=False)

In [ ]:
# Une agrégation ordinaire : rien de nouveau une fois les messages décodés
(actes.groupBy("region", "type_acte")
      .count()
      .orderBy(F.col("count").desc())
      .show(10))

### Mode flux : traiter en continu

Le même code, ou presque : `readStream` au lieu de `read`. La différence est
qu'aucune requête en flux ne se termine d'elle-même.

Deux garde-fous existent, et **le notebook les emploie systématiquement** :

- borner la durée avec `awaitTermination(secondes)` puis `stop()` ;
- ou demander à Spark de s'arrêter dès qu'il a tout lu, avec
  `trigger(availableNow=True)`.

In [ ]:
flux = (spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", COURTIER)
        .option("subscribe", SUJET)
        .option("startingOffsets", "earliest")
        .load()
        .select(F.from_json(F.col("value").cast("string"), schema).alias("a"))
        .select("a.*"))

# Premier garde-fou : traiter ce qui est disponible, puis s'arrêter seul
requete = (flux.groupBy("region")
           .count()
           .writeStream
           .outputMode("complete")
           .format("memory")
           .queryName("par_region")
           .trigger(availableNow=True)
           .start())

requete.awaitTermination()          # se termine seul grâce à availableNow
print("Requête terminée. Résultat :")
spark.sql("SELECT * FROM par_region ORDER BY count DESC").show()

### Écrire dans MongoDB, au fil de l'eau

C'est le premier maillon de la chaîne des projets : les événements arrivent par
Kafka, Spark les transforme, MongoDB les conserve.

In [ ]:
MONGO_URI = os.environ["MONGO_URI"]

# On relance un producteur en arrière-plan pour avoir du flux à traiter
import threading

def produire_en_continu(nombre=300, pause=0.02):
    p = Producer({"bootstrap.servers": COURTIER})
    for i in range(1001, 1001 + nombre):
        p.produce(SUJET, key=f"centre-{i % 7}",
                  value=json.dumps(evenement(i)).encode("utf-8"))
        p.poll(0)
        time.sleep(pause)
    p.flush()

threading.Thread(target=produire_en_continu, daemon=True).start()
print("Production lancée en arrière-plan.")

In [ ]:
def vers_mongo(lot, numero_lot):
    """Appelée pour chaque micro-lot : on y écrit ce que l'on veut."""
    (lot.write.format("mongodb")
        .option("connection.uri", MONGO_URI)
        .option("database", "flux")
        .option("collection", "actes")
        .mode("append")
        .save())
    print(f"  lot {numero_lot} : {lot.count()} actes écrits")


requete = (flux.writeStream
           .foreachBatch(vers_mongo)
           .option("checkpointLocation", "/tmp/checkpoint_actes")
           .start())

# Second garde-fou : durée bornée, puis arrêt propre
requete.awaitTermination(30)
requete.stop()
print("Requête arrêtée. Active :", requete.isActive)

`foreachBatch` est le point d'extension le plus utile de Spark en flux :
chaque micro-lot est un DataFrame ordinaire, sur lequel on fait ce que l'on
veut — écrire dans plusieurs destinations, appliquer une logique métier,
déclencher une alerte.

Le `checkpointLocation` mémorise où en est le traitement. Sans lui, un
redémarrage relirait tout depuis le début.

In [ ]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
collection = client["flux"]["actes"]
print(f"Documents dans MongoDB : {collection.count_documents({}):,}"
      .replace(",", " "))
for document in collection.find().limit(3):
    print({k: v for k, v in document.items() if k != "_id"})

## 6. Ce qu'il faut retenir

- Un sujet Kafka est un **journal conservé**, pas une file qui se vide : on peut
  relire, et plusieurs consommateurs lisent indépendamment.
- La **clé** du message détermine la partition, donc l'ordre garanti.
- Le **décalage** appartient au groupe de consommateurs : changer de groupe,
  c'est relire depuis le début.
- Spark lit Kafka en **lot** ou en **flux**. Le mode lot suffit très souvent, et
  il est plus simple à exploiter.
- Une requête en flux ne s'arrête jamais seule : borner la durée, ou employer
  `trigger(availableNow=True)`.
- `foreachBatch` traite chaque micro-lot comme un DataFrame ordinaire.
- Le `checkpointLocation` mémorise la progression : sans lui, tout est relu.

In [ ]:
spark.stop()
client.close()
print("Session arrêtée.")